# Designing an efficient 2-stage decimator 

**Topics:**
- Multistage decimation
- Computational efficiency
- Designing FIR filters using the frequency sampling method

**Learning goals:**
- Being able to calculate a desired frequency response, designing a filter and exporting said filter to a C program

**Libraries and notebook config:**

 

In [ ]:
from numpy import sin, cos, pi, exp, empty, mean, abs, angle # Sentrale matematiske funksjoner
import numpy as np                          # Importer funksjonalitet fra numpy biblioteket med prefiks "np"
import scipy.signal as sig                  # Importerer signalbehandlingsmodulen til scipy
import matplotlib.pyplot as plt             # Importer pyplot modulen i matplotlib med prefiks "plt"

plt.rcParams["figure.figsize"] = [8.00, 4.5]
plt.rcParams["figure.autolayout"] = True
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.xmargin"] = 0.0

%matplotlib inline

### System specifications

The particular oversampling system we want to implement in this lab is one where the input analog signal $x(t)$ is sampled at $f_{s, \small{\text{old}}} = 160\text{ KHz}$. Then, the signal is downsampled to $f_{s, \small{\text{new}}} = 10\text{ KHz}$ in two stages, with $M_1 = 8$ and $M_2 = 2$. A system bock diagram of the system is shown below:

![](figures/multistage_assignment_10.png)

We will start by setting up the filter for the first decimation stage.



## a)
Given that the end goal for our 2-stage decimation system is a signal $x_{\text{new}}[m]$ operating at a sampling rate $f_s = 10\ 000 \text{Hz}$, we should be able to formulate som ideal specifications for the decimation filter in stage $1$. Answer the following
* What is the ideal passband edge frequency $f_{\text{pass}}$ for the stage $1$ decimation filter?
* What is the ideal stopband edge frequency $f_{\text{stop}}$ for the stage $1$ decimation filter?

In [ ]:
f_pass = "???" # Fill in correct answer here
f_stop = "???" # Fill in correct answer here
# WRITE YOUR CODE IN THIS CELL:

In [ ]:
from hashlib import sha1
assert sha1(str(round(f_pass)).encode('utf-8')+b'e585b').hexdigest() == 'a70301a4678fed94f615c900c1f77d9c6bc5e11c', 'Wrong answer for f_pass :('; print('Correct answer for f_pass :)')
assert sha1(str(round(f_stop)).encode('utf-8')+b'35160').hexdigest() == '91b95e2060867c1fcc489b3cd0023a024345a7e7', 'Wrong answer for f_stop :('; print('Correct answer for f_stop :)')

A very easy way to downsample a signal which is widely used, is to simply calculate the sum of each block of $M_1 =8$ samples from $x_{\text{old}}[n]$, with the new "sum-of-block" signal being the intermediary signal $x_{\text{int}}[l]$ between stages $1$ and $2$. 

\begin{align*}
x_{\text{int}}[l] & = \begin{cases}
x_{\text{old}}[0] + x_{\text{old}}[1] + x_{\text{old}}[2] + \ldots + x_{\text{old}}[7], & l = 0 \\
x_{\text{old}}[8] + x_{\text{old}}[9] + x_{\text{old}}[10] + \ldots + x_{\text{old}}[15], & l = 1 \\
x_{\text{old}}[16] + x_{\text{old}}[17] + x_{\text{old}}[18] + \ldots + x_{\text{old}}[23], & l = 2 \\
\qquad \qquad \vdots, & \vdots
\end{cases} \\ \tag{1} \\
&= x_{\text{old}}[8 \cdot l] + x_{\text{old}}[8 \cdot l + 1] + x_{\text{old}}[8 \cdot l + 2] + \ldots + x_{\text{old}}[8 \cdot l + 7]
\end{align*}

Below is a code cell showing how this process may be implemented in Python:

In [ ]:
import numpy as np
def decimator_stage_1(x: np.ndarray, M: int):
    """
    Decimates a signal x[n] represented by the input array 'x' with 
    decimation factor M. Each sample in the decimated output y[l] 
    equals the sum of M neighboring samples from the input signal x[n].
    """
    L_x = len(x)
    L_y = L_x//M
    
    y = np.zeros(L_y)
    for l in range(L_y):
        accumulator = 0
        for n in range(M):
            accumulator += x[M*l+n]
        y[l] = accumulator
    return y

The process described in equation $1$ as well as the above code cell, can be modeled as a $M_1=8$-point moving average filter (without the $\frac{1}{8}$ term) followed by downsampling by a factor $8$.

![](figures/stage_1.png)

## b)

* Use Python to generate a test signal of your choice with length $L=128$, and perform factor $M_1=8$ decimation of the signal using:
    1. A filter `hn = np.ones(8)` followed by downsampling using slicing (like we learned in assignment 9)
    2. The "sum-of-block" approach implemented in the function `downsampler_stage_1` which combines both filtering.
* Compare the outputs for both these approaches, and ensure they are identical.

In [ ]:
# WRITE YOUR CODE IN THIS CELL:

Now that we have established how stage $1$ can be modeled in terms of a decimation filter and downsampling operation, we should analyze our decimation filter and see how it holds up to our expectations for a decimation filter.

## c) 

* Calculate the frequency response $H_1(\hat{\omega})$ for the decimation filter $h_1 = u[n] - u[n-8]$, and show the magnitude response of the filter as a function of physical frequency $f$ in Hertz. In the same figure, highlight the magnitude response for the filter in the ideal passband frequency range from problem **a)**.

In [ ]:
# WRITE YOUR CODE IN THIS CELL:

[Here](figures/task1c_sol.svg) is a picture showing what the magnitude response $|H_1(f)|$ might look like. As we can see from the magnitude response plotted in problem **c)**, the passband is not flat, but droops noticeably towards $f_{\text{pass}}$. No need to worry too much though, because we can address this issue in stage two of our multistage decimation system.

#### Stage 2:

We can compensate for the drooping passband by tuning the decimation filter in stage 2 to act as a *compensation FIR filter* which ensures that the full cascaded response of filter $1$ and filter $2$ is approximately flat for the target passband. The ideal response of such a filter is:

\begin{equation*}
|H_2(f)| = \begin{cases}
\frac{1}{|H_1(f)|}, & |f| < f_{\text{pass}} \\
0, & |f| \geq f_{\text{pass}}
\end{cases}\tag{2}
\end{equation*}

where $f_{\text{pass}}$ is the ideal passband edge frequency found in problem **a)**. To design such a filter, the frequency sampling method is highly suitable as it easily allows for calculating the approximate impulse response of a filter with any shape that can be represented by an array of magnitude values.

## d)

Find the approximate impulse response for $h_2[l]$ as a $129$-point FIR filter by performing the following steps:

1. Calculate the magnitude response $|H_1(\hat{\omega})|$ of filter $h_1[n]$ in $\frac{257+1}{2}=129$ linearly spaced points along the frequency axis at intervals $\frac{f_{s, \text{int}}}{257}$<br>*Hint:* `f, Hf = sig.freqz(hn, worN=np.arange(0, f_max, delta_f), fs=fs)`
  
2. Calculate the magnitude response $|H_2(\hat{\omega})|$ according to equation $2$ above.

3. Use `numpy.fft.irfft` to transform $H_2(\hat{\omega})$ to the time domain as $h_2[l]$.

4. Introduce a sample delay of $D=\frac{257-1}{2} = 128$ samples to the impulse response $h_2[l]$ by applying a circular shift of $128$ points toward the right.<br>*Hint:* `numpy.roll`

5. Perform frequency-domain smoothing by applying the window function of your choice to the impulse response $h_2[l]$

6. Plot the final frequency magnitude response and impulse response in separate subplots

In [ ]:
h1n = np.ones(8)
numtaps = 129
# WRITE YOUR CODE IN THIS CELL:

[Here](figures/task1d_sol.svg) is a picture showing what the final filter should look like.

The final step needed before we can head to programming the STM32 is to export our FIR filter coefficients to a header file. If you don't immediately know how, I would suggest this is a prime example of a situation where LLM's can come in handy as this is a fairly straightforward programming problem which has little to do with actual DSP theory and the program's output is easily verified. 

## e)
* Use your AI tool of choice to write a program which exports the array with impulse response values $h_2[n]$ to a header file using `float32_t` representation.

In [ ]:
# WRITE YOUR CODE IN THIS CELL:

#### Some noteworthy context:

What we have designed in this part of the assignment is more commonly known as a first-order Cascaded Integrator-Comb (CIC) Decimator with FIR compensation filter. CIC filters are often used in oversampling due to the light processing load they require, and can be further adjusted to achieve greater stopband attuenuation by using a second-order CIC (or higher). If you are curious to know more about these filters, you can read about them in chapter $10.14$ in the book Understanding Digital Signal Processing.